# Sampled label consistency

This notebook reads the sampled-label export from `data/labels_sampled/`, merges it with the original labels in `data/labels.jsonl`, and measures agreement on the 1-4 execution score using Cohen's weighted kappa.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
LABELS_SAMPLED_DIR = DATA_DIR / "labels_sampled"
LABELS_PATH = DATA_DIR / "labels.jsonl"

pd.set_option("display.max_colwidth", 120)

In [ ]:
def extract_first_choice(results, result_type, value_key):
    for result in results:
        if result.get("type") != result_type:
            continue
        value = result.get("value", {}).get(value_key)
        if isinstance(value, list):
            return value[0] if value else None
        return value
    return None


sampled_rows = []
for path in sorted(LABELS_SAMPLED_DIR.iterdir()):
    row = json.loads(path.read_text())
    results = row.get("result", [])
    video_uri = row.get("task", {}).get("data", {}).get("video")
    clip_name = Path(video_uri.split("://", 1)[-1]).name if video_uri else None
    sampled_rows.append(
        {
            "sampled_export_id": row.get("id"),
            "sampled_task_id": row.get("task", {}).get("id"),
            "clip_name": clip_name,
            "sampled_video_uri": video_uri,
            "sampled_created_at": row.get("created_at"),
            "sampled_completed_by": row.get("completed_by", {}).get("email"),
            "sampled_trick": extract_first_choice(results, "choices", "choices"),
            "sampled_score": extract_first_choice(results, "rating", "rating"),
        }
    )

sampled_df = pd.DataFrame(sampled_rows)
sampled_df.head()

In [ ]:
original_df = pd.read_json(LABELS_PATH, lines=True)
original_df["clip_name"] = original_df["video"].map(lambda uri: Path(uri).name)
original_df = original_df.rename(
    columns={
        "video": "original_video_uri",
        "trick": "original_trick",
        "execution_score": "original_score",
        "created_at": "original_created_at",
        "updated_at": "original_updated_at",
        "annotator": "original_annotator",
        "annotation_id": "original_annotation_id",
    }
)

merged_df = sampled_df.merge(
    original_df[
        [
            "clip_name",
            "original_video_uri",
            "original_trick",
            "original_score",
            "original_explanation",
            "original_created_at",
            "original_updated_at",
            "original_annotator",
            "original_annotation_id",
            "person",
            "key_foot",
        ]
    ],
    on="clip_name",
    how="left",
)

merged_df["score_delta"] = merged_df["sampled_score"] - merged_df["original_score"]
merged_df["absolute_score_delta"] = merged_df["score_delta"].abs()
merged_df["exact_match"] = merged_df["sampled_score"] == merged_df["original_score"]

print(f"Sampled export rows: {len(sampled_df)}")
print(f"Matched original labels: {merged_df['original_score'].notna().sum()}")
print(
    f"Sampled rows with trick labels present: {merged_df['sampled_trick'].notna().sum()}"
)

merged_df.head()

In [ ]:
agreement_df = merged_df.dropna(subset=["sampled_score", "original_score"]).copy()
agreement_df[["sampled_score", "original_score"]] = agreement_df[
    ["sampled_score", "original_score"]
].astype(int)

dummy_mode_class = int(agreement_df["original_score"].mode().iloc[0])
agreement_df["dummy_score"] = dummy_mode_class

# Quadratic weights are the primary agreement metric because 1-4 stars are ordinal.
summary_df = pd.DataFrame(
    [
        {"metric": "sampled_export_rows", "value": len(sampled_df)},
        {"metric": "matched_rows", "value": len(agreement_df)},
        {"metric": "dummy_mode_class", "value": dummy_mode_class},
        {
            "metric": "exact_agreement",
            "value": float(
                (agreement_df["sampled_score"] == agreement_df["original_score"]).mean()
            ),
        },
        {
            "metric": "mean_absolute_score_delta",
            "value": float(agreement_df["absolute_score_delta"].mean()),
        },
        {
            "metric": "dummy_mae_vs_original",
            "value": float(
                (agreement_df["original_score"] - agreement_df["dummy_score"])
                .abs()
                .mean()
            ),
        },
        {
            "metric": "dummy_mae_vs_sampled",
            "value": float(
                (agreement_df["sampled_score"] - agreement_df["dummy_score"])
                .abs()
                .mean()
            ),
        },
        {
            "metric": "cohen_kappa_unweighted",
            "value": float(
                cohen_kappa_score(
                    agreement_df["original_score"], agreement_df["sampled_score"]
                )
            ),
        },
        {
            "metric": "cohen_kappa_weighted_linear",
            "value": float(
                cohen_kappa_score(
                    agreement_df["original_score"],
                    agreement_df["sampled_score"],
                    weights="linear",
                )
            ),
        },
        {
            "metric": "cohen_kappa_weighted_quadratic",
            "value": float(
                cohen_kappa_score(
                    agreement_df["original_score"],
                    agreement_df["sampled_score"],
                    weights="quadratic",
                )
            ),
        },
        {
            "metric": "dummy_qwk_vs_original",
            "value": float(
                cohen_kappa_score(
                    agreement_df["original_score"],
                    agreement_df["dummy_score"],
                    weights="quadratic",
                )
            ),
        },
        {
            "metric": "dummy_qwk_vs_sampled",
            "value": float(
                cohen_kappa_score(
                    agreement_df["sampled_score"],
                    agreement_df["dummy_score"],
                    weights="quadratic",
                )
            ),
        },
    ]
)

score_confusion_df = pd.crosstab(
    agreement_df["original_score"],
    agreement_df["sampled_score"],
    rownames=["original_score"],
    colnames=["sampled_score"],
    margins=True,
)

score_delta_df = (
    agreement_df["score_delta"]
    .value_counts()
    .sort_index()
    .rename_axis("score_delta")
    .to_frame("count")
)

score_distribution_df = pd.DataFrame({"score": [1, 2, 3, 4]})
score_distribution_df["original_count"] = (
    score_distribution_df["score"]
    .map(agreement_df["original_score"].value_counts())
    .fillna(0)
    .astype(int)
)
score_distribution_df["sampled_count"] = (
    score_distribution_df["score"]
    .map(agreement_df["sampled_score"].value_counts())
    .fillna(0)
    .astype(int)
)
score_distribution_df["original_pct"] = (
    score_distribution_df["original_count"] / len(agreement_df)
).round(4)
score_distribution_df["sampled_pct"] = (
    score_distribution_df["sampled_count"] / len(agreement_df)
).round(4)

display(summary_df)
display(score_distribution_df)
display(score_confusion_df)
display(score_delta_df)

In [ ]:
disagreements_df = merged_df[~merged_df["exact_match"]].copy()
disagreements_df = disagreements_df.sort_values(
    ["absolute_score_delta", "clip_name"],
    ascending=[False, True],
)

display(
    disagreements_df[
        [
            "clip_name",
            "person",
            "original_trick",
            "original_score",
            "sampled_score",
            "score_delta",
            "original_explanation",
        ]
    ]
)